# Sana-0.6B BSS/BDS Colab runner

This notebook follows the same path and git-pull style as `model_a_relitlive_official_bss_run_all.ipynb`: clone/pull code into `/content`, keep run outputs under a local run root, mirror heavy artifacts to Drive, and only run the mini-suite after smoke passes.

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/WANG-Ruipeng/Sana.git"
BRANCH = "sana06b-bss-bds"
REPO_DIR = Path("/content/Sana-BSS")
SANA_UPSTREAM_URL = "https://github.com/NVlabs/Sana.git"
SANA_UPSTREAM_DIR = Path("/content/Sana")

RUN_NAME = "sana06b_bss_bds_v1"
RUNS_ROOT = Path("/content/Sana-BSS-Runs")
DRIVE_RUNS_ROOT = Path("/content/drive/MyDrive/Colab_Projects/Sana-BSS-BDS")
RUN_ROOT = RUNS_ROOT / RUN_NAME
DRIVE_RUN_ROOT = DRIVE_RUNS_ROOT / RUN_NAME

# Smoke first. Only set RUN_MINI_SUITE=True after smoke passes.
RUN_SMOKE = True
RUN_MINI_SUITE = False

# Install repo/minimal dependencies before smoke/full inference.
INSTALL_DEPS = True
RUN_ENV_SETUP = False

# Set True only if you want to discard the existing /content/Sana-BSS checkout.
FORCE_RECLONE = False
MOUNT_DRIVE = True

# Large weights are not stored in git.
DRIVE_WEIGHTS_ROOT = Path("/content/drive/MyDrive/ModelWeights/Sana")
WEIGHTS_SOURCE_DIR = DRIVE_WEIGHTS_ROOT / "Sana_600M_1024px"
WEIGHTS_RUN_DIR = WEIGHTS_SOURCE_DIR

# Optional fallback. Keep False unless you really want Colab to download large model files from Hugging Face.
AUTO_DOWNLOAD_FROM_HF = False
HF_MODEL_ID = "Efficient-Large-Model/Sana_600M_1024px"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

BACKEND = "native"
CONFIG_PATH = "configs/sana_config/1024ms/Sana_600M_img1024.yaml"
SCRIPT_ROOT = REPO_DIR / "bss_experiments/sana06b_bss_bds_v1/scripts"

print("REPO_URL:", REPO_URL)
print("BRANCH:", BRANCH)
print("REPO_DIR:", REPO_DIR)
print("RUN_ROOT:", RUN_ROOT)
print("DRIVE_RUN_ROOT:", DRIVE_RUN_ROOT)
print("RUN_SMOKE:", RUN_SMOKE)
print("RUN_MINI_SUITE:", RUN_MINI_SUITE)
print("WEIGHTS_SOURCE_DIR:", WEIGHTS_SOURCE_DIR)


In [ ]:
import getpass
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount skipped or failed:", repr(exc))

RUN_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Runtime setup done.")


In [ ]:
def _display_cmd(cmd):
    text = " ".join(str(part) for part in cmd)
    if "x-access-token:" in text:
        text = text.split("x-access-token:")[0] + "x-access-token:***@" + text.split("@", 1)[-1]
    return text


def run(cmd, cwd=None, check=True):
    print("$", _display_cmd(cmd))
    return subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=check, text=True)


def authenticated_url(url):
    token = os.environ.get("GITHUB_TOKEN")
    if token is None:
        token = getpass.getpass("GitHub token for clone/fetch; leave blank if the branch is public: ")
        if token:
            os.environ["GITHUB_TOKEN"] = token
    if token:
        return url.replace("https://", f"https://x-access-token:{token}@", 1)
    return url


def checkout_repo(repo_url, branch, repo_dir, *, force_reclone=False, single_branch=True):
    clone_url = authenticated_url(repo_url)
    repo_dir = Path(repo_dir)

    if force_reclone and repo_dir.exists():
        shutil.rmtree(repo_dir)

    if repo_dir.exists() and not (repo_dir / ".git").exists():
        fallback = repo_dir.with_name(f"{repo_dir.name}_non_git_{int(time.time())}")
        print(f"Existing non-git directory found at {repo_dir}; moving it to {fallback}")
        shutil.move(str(repo_dir), str(fallback))

    if not repo_dir.exists():
        cmd = ["git", "clone"]
        if single_branch:
            cmd += ["--branch", branch, "--single-branch"]
        cmd += [clone_url, str(repo_dir)]
        run(cmd)
    else:
        run(["git", "-C", str(repo_dir), "remote", "set-url", "origin", clone_url])
        run(["git", "-C", str(repo_dir), "fetch", "origin", branch])
        run(["git", "-C", str(repo_dir), "switch", "-C", branch, f"origin/{branch}"])
        run(["git", "-C", str(repo_dir), "reset", "--hard", f"origin/{branch}"])

    # Avoid leaving a token-bearing URL in .git/config.
    run(["git", "-C", str(repo_dir), "remote", "set-url", "origin", repo_url])
    run(["git", "-C", str(repo_dir), "branch", "--show-current"])
    run(["git", "-C", str(repo_dir), "rev-parse", "HEAD"])
    run(["git", "-C", str(repo_dir), "status", "--short"])


checkout_repo(REPO_URL, BRANCH, REPO_DIR, force_reclone=FORCE_RECLONE, single_branch=True)

# Keep an official upstream checkout for audit/comparison, but the experiment code runs from REPO_DIR.
if not SANA_UPSTREAM_DIR.exists():
    run(["git", "clone", SANA_UPSTREAM_URL, str(SANA_UPSTREAM_DIR)])
else:
    run(["git", "-C", str(SANA_UPSTREAM_DIR), "fetch", "origin", "main"], check=False)
    run(["git", "-C", str(SANA_UPSTREAM_DIR), "switch", "main"], check=False)
    run(["git", "-C", str(SANA_UPSTREAM_DIR), "reset", "--hard", "origin/main"], check=False)
run(["git", "-C", str(SANA_UPSTREAM_DIR), "rev-parse", "HEAD"], check=False)


In [ ]:
os.environ["PYTHONPATH"] = f"{REPO_DIR}:{SANA_UPSTREAM_DIR}:" + os.environ.get("PYTHONPATH", "")

if INSTALL_DEPS:
    if RUN_ENV_SETUP:
        run(["bash", "environment_setup.sh", "sana"], cwd=REPO_DIR)
    else:
        run([
            sys.executable, "-m", "pip", "install", "-q",
            "diffusers>=0.32.0", "transformers", "accelerate", "safetensors",
            "sentencepiece", "huggingface_hub", "opencv-python", "imageio",
            "pandas", "numpy", "pillow", "matplotlib", "omegaconf", "pyrallis", "termcolor"
        ])
    print("Dependency install/check completed.")
else:
    print("INSTALL_DEPS=False; skipping dependency installation.")


In [ ]:
def sana_weights_valid(path):
    path = Path(path)
    candidates = [
        path / "checkpoints/Sana_600M_1024px.pth",
        path / "checkpoint/Sana_600M_1024px.pth",
        path / "Sana_600M_1024px.pth",
    ]
    return path.is_dir() and (any(p.exists() for p in candidates) or any(path.rglob("*.pth")))

print("WEIGHTS_SOURCE_DIR:", WEIGHTS_SOURCE_DIR)
print("weights valid:", sana_weights_valid(WEIGHTS_SOURCE_DIR))
if WEIGHTS_SOURCE_DIR.exists():
    print("pth files:", [str(p) for p in WEIGHTS_SOURCE_DIR.rglob("*.pth")][:8])

if AUTO_DOWNLOAD_FROM_HF and not sana_weights_valid(WEIGHTS_SOURCE_DIR):
    WEIGHTS_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
    cmd = ["huggingface-cli", "download", HF_MODEL_ID, "--local-dir", str(WEIGHTS_SOURCE_DIR)]
    if HF_TOKEN:
        cmd += ["--token", HF_TOKEN]
    run(cmd)

if not sana_weights_valid(WEIGHTS_SOURCE_DIR):
    print("Missing Sana-0.6B weights. Put them on Drive or set AUTO_DOWNLOAD_FROM_HF=True.")
    print(f"Expected: {WEIGHTS_SOURCE_DIR}")
    print(f"Example: huggingface-cli download {HF_MODEL_ID} --local-dir {WEIGHTS_SOURCE_DIR}")
    raise FileNotFoundError("Missing Sana-0.6B weights")


In [ ]:
required_paths = {
    "experiment scripts": SCRIPT_ROOT,
    "adapter": SCRIPT_ROOT / "sana06b_adapter.py",
    "native pipeline": REPO_DIR / "app/sana_pipeline.py",
    "600M config": REPO_DIR / CONFIG_PATH,
    "weights dir": WEIGHTS_RUN_DIR,
}

missing = []
for label, path in required_paths.items():
    ok = sana_weights_valid(path) if label == "weights dir" else Path(path).exists()
    print(f"{label:24s} {'OK' if ok else 'MISSING'}  {path}")
    if not ok:
        missing.append((label, path))

if missing:
    for label, path in missing:
        print(f"- {label}: {path}")
    raise FileNotFoundError("Missing required Sana BSS/BDS paths")

os.environ.update({
    "DRIVE_WEIGHTS_ROOT": str(DRIVE_WEIGHTS_ROOT),
    "DRIVE_EXPERIMENT_ROOT": str(DRIVE_RUN_ROOT),
    "SANA06B_WEIGHTS_DIR": str(WEIGHTS_RUN_DIR),
})

print("Preflight paths look ready.")


In [ ]:
run([sys.executable, str(SCRIPT_ROOT / "make_manifest_sana06b_bds.py"), "--experiment-root", str(RUN_ROOT), "--backend", BACKEND], cwd=REPO_DIR)
run([sys.executable, str(SCRIPT_ROOT / "audit_sana06b.py"), "--weights-dir", str(WEIGHTS_RUN_DIR), "--output", str(RUN_ROOT / "reports/00_repo_model_hardware_audit.md")], cwd=REPO_DIR)
run([sys.executable, str(SCRIPT_ROOT / "validate_schedules.py"), "--output", str(RUN_ROOT / "schedules/schedule_validation.json")], cwd=REPO_DIR)


In [ ]:
def run_stream(cmd, cwd=REPO_DIR):
    print("$", _display_cmd(cmd))
    env = os.environ.copy()
    proc = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}: {_display_cmd(cmd)}")

if RUN_SMOKE:
    run_stream([
        sys.executable, str(SCRIPT_ROOT / "run_manifest.py"),
        "--manifest", str(RUN_ROOT / "manifests/sana06b_smoke_manifest.csv"),
        "--experiment-root", str(RUN_ROOT),
        "--backend", BACKEND,
        "--weights-dir", str(WEIGHTS_RUN_DIR),
        "--config-path", CONFIG_PATH,
        "--resume",
        "--sync_drive",
        "--drive-experiment-root", str(DRIVE_RUN_ROOT),
    ])
    run([sys.executable, str(SCRIPT_ROOT / "compute_metrics_against_ref.py"), "--manifest", str(RUN_ROOT / "manifests/sana06b_smoke_manifest.csv"), "--experiment-root", str(RUN_ROOT)], cwd=REPO_DIR)
    run([sys.executable, str(SCRIPT_ROOT / "generate_figures.py"), "--manifest", str(RUN_ROOT / "manifests/sana06b_smoke_manifest.csv"), "--experiment-root", str(RUN_ROOT)], cwd=REPO_DIR)
else:
    print("RUN_SMOKE=False; smoke was not started.")


In [ ]:
if RUN_MINI_SUITE:
    run_stream([
        sys.executable, str(SCRIPT_ROOT / "run_manifest.py"),
        "--manifest", str(RUN_ROOT / "manifests/sana06b_prompt_suite_manifest.csv"),
        "--experiment-root", str(RUN_ROOT),
        "--backend", BACKEND,
        "--weights-dir", str(WEIGHTS_RUN_DIR),
        "--config-path", CONFIG_PATH,
        "--resume",
        "--sync_drive",
        "--drive-experiment-root", str(DRIVE_RUN_ROOT),
    ])
    run([sys.executable, str(SCRIPT_ROOT / "compute_metrics_against_ref.py"), "--manifest", str(RUN_ROOT / "manifests/sana06b_prompt_suite_manifest.csv"), "--experiment-root", str(RUN_ROOT)], cwd=REPO_DIR)
    run([sys.executable, str(SCRIPT_ROOT / "compute_bds.py"), "--metrics-dir", str(RUN_ROOT / "metrics")], cwd=REPO_DIR)
    run([sys.executable, str(SCRIPT_ROOT / "generate_figures.py"), "--manifest", str(RUN_ROOT / "manifests/sana06b_prompt_suite_manifest.csv"), "--experiment-root", str(RUN_ROOT)], cwd=REPO_DIR)
    run([sys.executable, str(SCRIPT_ROOT / "write_final_report.py"), "--experiment-root", str(RUN_ROOT)], cwd=REPO_DIR)
else:
    print("RUN_MINI_SUITE=False, so the 16-prompt mini-suite was not started.")


In [ ]:
important_paths = {
    "experiment folder": RUN_ROOT,
    "Drive mirror": DRIVE_RUN_ROOT,
    "audit report": RUN_ROOT / "reports/00_repo_model_hardware_audit.md",
    "smoke manifest": RUN_ROOT / "manifests/sana06b_smoke_manifest.csv",
    "mini manifest": RUN_ROOT / "manifests/sana06b_prompt_suite_manifest.csv",
    "metrics CSV": RUN_ROOT / "metrics/master_long_metrics.csv",
    "BDS table": RUN_ROOT / "tables/tableA_sana06b_bds_by_split.md",
    "paper row": RUN_ROOT / "tables/table_cross_model_same_compute_sana06b_row.md",
    "LaTeX row": RUN_ROOT / "tables/table_cross_model_same_compute_sana06b_row.tex",
    "final report": RUN_ROOT / "reports/FINAL_SANA06B_BSS_BDS_REPORT.md",
    "side-by-side index": RUN_ROOT / "figures/side_by_side/index.html",
}

for label, path in important_paths.items():
    print(f"{label:24s} {'OK' if path.exists() else 'not yet'}  {path}")

paper_row = RUN_ROOT / "tables/table_cross_model_same_compute_sana06b_row.md"
if paper_row.exists():
    print("\n===== Paper-format Row =====")
    print(paper_row.read_text(encoding="utf-8"))

final_report = RUN_ROOT / "reports/FINAL_SANA06B_BSS_BDS_REPORT.md"
if final_report.exists():
    print("\n===== Final Report Path =====")
    print(final_report)
